In [ ]:
import json
import time
import re
import unicodedata
import re
import unidic2ud
import os
import glob
from ufal.udpipe import Model, Pipeline

In [ ]:
model_path = '/opt/anaconda3/envs/udpipe_env/lib/python3.10/site-packages/unidic2ud/download/japanese-modern.udpipe'

model = Model.load(model_path)

pipeline = Pipeline(
    model,
    "tokenize",
    Pipeline.DEFAULT,
    Pipeline.DEFAULT,
    "conllu"
)


In [ ]:
lines = []
with open("jpn_news_2023_30K-sentences.txt", "r", encoding="utf-8") as f:
    for line in f:
        cleaned_line = line.strip()
        
        text = re.sub(r"^\d+\s*", "", cleaned_line)

        lines.append(text)
lines = lines[1:500]

In [ ]:
start = time.perf_counter()
for line in lines:
    processed = pipeline.process(line)

end = time.perf_counter()

print(end - start)

In [ ]:
all_tokens_udpipe = []
clean_tokens_udpipe = []
all_pos_udpipe = []         

for line in lines:
    processed = pipeline.process(line)
    
    for l in processed.split('\n'):        
        if l.startswith('#') or not l.strip():
            continue
        
        parts = l.split('\t')              
        if len(parts) < 4:                 
            continue
        
        token = parts[1]
        upos  = parts[3]                   
        
        norm = unicodedata.normalize('NFKC', token)
        
        all_tokens_udpipe.append(norm)
        all_pos_udpipe.append(upos)        
        
        if norm.isdigit():
            continue
        
        if re.match(r'^\W+$', norm):      
            continue
        
        clean_tokens_udpipe.append(norm)

In [ ]:
data = {
    "all_tokens": all_tokens_udpipe,
    "clean_tokens": clean_tokens_udpipe,
    "pos": all_pos_udpipe
}

with open("udpipe_tokens.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)

In [ ]:
print(len(all_pos_udpipe))

In [ ]:
sum(len(t) for t in all_tokens_udpipe) / len(all_tokens_udpipe)



In [ ]:
def parse_udpipe_lines(lines):
    results = []
    
    for line in lines:
        if not line.strip():
            continue
        
        result = pipeline.process(line)
        results.append(result)
    
    return results

In [ ]:
udpipe_result = parse_udpipe_lines(lines)

In [ ]:
def extract_udpipe_dependencies(sent):
    deps = []

    for line in sent.splitlines():
        if line.startswith("#") or not line.strip():
            continue

        parts = line.split("\t")
        if len(parts) < 8:
            continue

        head = parts[6]   # HEAD
        dep = parts[0]    # ID
        rel = parts[7]    # DEPREL

        if head != "0":
            deps.append((dep, head, rel))

    return deps

In [ ]:
def extract_udpipe_all_dependencies(udpipe_results):
    all_deps = []
    
    for sent in udpipe_results:
        deps = extract_udpipe_dependencies(sent)
        if deps:
            all_deps.append(deps)
    
    return all_deps

In [ ]:
udpipe_result_dependencies = extract_udpipe_all_dependencies(udpipe_result)
print(sum(len(deps) for deps in udpipe_result_dependencies))

In [ ]:
def longest_dependency_per_sentence_with_text(sentences):
    results = []

    for sent_id, line in enumerate(sentences):
        if not line.strip():
            continue

        deps = []
        text = ""
        lines = line.strip().split('\n')

        # Парсим CoNLL-U формат UDPipe
        for i, ln in enumerate(lines):
            if ln.startswith('# text ='):
                text = ln.split('=')[1].strip()
                continue

            if ln.startswith('#') or not ln.strip():
                continue

            parts = ln.split('\t')
            if len(parts) < 8:
                continue

            # ID, FORM, LEMMA, UPOS, XPOS, FEATS, HEAD, DEPREL
            try:
                token_id = int(parts[0])
                head_id = int(parts[6]) if parts[6] != '_' else -1
                deprel = parts[7]
                deps.append((token_id, head_id, deprel))
            except (ValueError, IndexError):
                continue

        # Ищем самую длинную зависимость
        max_dist = 0
        max_edge = None

        for dep, head, rel in deps:
            if head == 0 or head == -1:  # root или пропуск
                continue
            dist = abs(dep - head)
            if dist > max_dist:
                max_dist = dist
                max_edge = (dep, head, rel)

        results.append({
            "sentence_id": sent_id,
            "text": text or f"Sentence {sent_id}",
            "max_distance": max_dist,
            "edge": max_edge
        })

    return results

In [ ]:
data = parse_udpipe_lines(lines)

longest = longest_dependency_per_sentence_with_text(data)

worst = max(longest, key=lambda x: x["max_distance"])

In [ ]:
print(worst)

In [ ]:
w ='２日間とも１６人出席し忌憚なく意見交換し出された意見・要望に関し、達専務理事は「人手不足、物価高騰、物流などの２０２４年問題など全ての事柄について国や道、稚内市に対し会員の皆さんの声を伝えて行きたい」と丁寧に対処すること約束していた。'

def interpret_edge(edge, tokens):
    dep_id, head_id, rel = edge

    dep_word = tokens.get(dep_id, "?")
    head_word = tokens.get(head_id, "?")

    return {
        "dependent": dep_word,
        "head": head_word,
        "relation": rel
    }

def extract_deps_conllu(conllu_text):
    deps = []
    tokens = {}

    for line in conllu_text.splitlines():
        if line.startswith("#") or not line.strip():
            continue

        parts = line.split("\t")
        if len(parts) < 8:
            continue

        idx = parts[0]
        form = parts[1]
        head = parts[6]
        rel = parts[7]

        if "-" in idx:
            continue

        tokens[idx] = form
        deps.append((idx, head, rel))

    return deps, tokens

doc = pipeline.process(w)

deps, tokens = extract_deps_conllu(doc)

interpret_edge(('21', '71', 'advcl'), tokens)

In [ ]:
def average_dependencies_per_sentence(all_sent_deps):
    total_deps = 0
    num_sentences = 0

    for deps in all_sent_deps:
        if not deps:
            continue

        total_deps += len(deps)
        num_sentences += 1

    return total_deps / num_sentences if num_sentences > 0 else 0


avg = average_dependencies_per_sentence(udpipe_result_dependencies)
print(round(avg,2))

In [ ]:
sample = lines[72]

In [ ]:
from collections import defaultdict

def build_tree(deps, tokens):
    children = defaultdict(list)
    rels = {}
    root = None

    for dep, head, rel in deps:
        rels[dep] = rel

        if head == "0":
            root = dep
        else:
            children[head].append(dep)

    return root, children, rels, tokens

In [ ]:
def print_tree(node, children, rels, tokens, prefix=""):
    word = tokens.get(node, "ROOT")
    rel = rels.get(node, "root")

    print(prefix + f"{word} [{rel}]")

    child_list = children.get(node, [])

    for i, child in enumerate(child_list):
        is_last = (i == len(child_list) - 1)

        if is_last:
            new_prefix = prefix + "   "
            branch = "└── "
        else:
            new_prefix = prefix + "│  "
            branch = "├── "

        print(prefix + branch, end="")
        print_tree(child, children, rels, tokens, new_prefix)

In [ ]:
text = "１２年余にわたる拉致監禁から解放された後、後藤さんが駆け込んだのはマンションの目と鼻の先にある交番だった。"

doc = pipeline.process(text)

deps, tokens = extract_deps_conllu(doc)

root, children, rels, tokens = build_tree(deps, tokens)

print_tree(root, children, rels, tokens)

In [ ]:
print(len(deps))